In [1]:
# Load Data

import pandas as pd

df = pd.read_csv("../data/credit_risk_dataset.csv")

In [2]:
# Exploratory Data Analysis

print("Shape of data:")
print(df.shape)

print("\nFirst 5 rows of dataset:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nNumerical Columns:")
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(numeric_cols)
      
print("\nCategorial Columns")
categorical_cols = df.select_dtypes(include=['category', 'object', 'string']).columns.tolist()
print(categorical_cols)

print("\nUnique values:")
print(df.nunique())

Shape of data:
(32581, 12)

First 5 rows of dataset:
   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59    

In [3]:
# Exploratory Data Analysis (cont.)

print("\nMissing values:")
print(df.isnull().sum())

# Select only the numeric columns
numeric_df = df.select_dtypes(include=['number'])

# Calculate mean, median, mode, and range
means = numeric_df.mean()
medians = numeric_df.median()
modes = numeric_df.mode().iloc[0] 
ranges = numeric_df.max() - numeric_df.min()

# Combine results into a clean, readable DataFrame
summary_df = pd.DataFrame({
    'Mean': means,
    'Median': medians,
    'Mode': modes,
    'Range': ranges
})
print("\nStatistical Metrics:")
print(summary_df)


Missing values:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Statistical Metrics:
                                    Mean    Median      Mode       Range
person_age                     27.734600     26.00     23.00      124.00
person_income               66074.848470  55000.00  60000.00  5996000.00
person_emp_length               4.789686      4.00      0.00      123.00
loan_amnt                    9589.371106   8000.00  10000.00    34500.00
loan_int_rate                  11.011695     10.99      7.51       17.80
loan_status                     0.218164      0.00      0.00        1.00
loan_percent_inco

In [4]:
# Exploratory Data Analysis (cont.)

print("\nTarget distribution:")
print(df["loan_status"].value_counts())

print("\nTarget percentage:")
print(df["loan_status"].value_counts(normalize=True))


Target distribution:
loan_status
0    25473
1     7108
Name: count, dtype: int64

Target percentage:
loan_status
0    0.781836
1    0.218164
Name: proportion, dtype: float64


In [5]:
# Hypotheses

# Hypothesis 1: A person's income is stongly correlated with loan status.

# Hypothesis 2: Higher loan amounts may be associated with higher default rate.

# Hypothesis 3: Someone who has defaulted before may be more likely to default again.

# Hypothesis 4: Loan amount as a ratio of annual income is the strongest factor when determining default rate.

In [6]:
# Data Cleaning

import pandas as pd
import numpy as np

# 1. Clean clear data errors
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] <= 60]

# 2. Impute employment length using median
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

# 3. Impute interest rate using loan grade medians
df['loan_int_rate'] = df.groupby('loan_grade')['loan_int_rate'].transform(
    lambda x: x.fillna(x.median())
)

# 4. Transform skewed income feature
df['person_income_log'] = np.log1p(df['person_income'])

In [11]:
# Exploratory Data Analysis (cont.)

# Categorical features
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

# Loop through each categorical feature
for col in cat_cols:
    summary = df.groupby(col)['loan_status'].agg(
        total_borrowers='count',
        default_count='sum',
        default_rate='mean'
    ).reset_index()
    
    # Format default rate as percentage
    summary['default_rate'] = (summary['default_rate'] * 100).round(2).astype(str) + '%'
    
    print(f"--- Groupby: {col} ---")
    print(summary)
    print("\n")

# 1. Binning numerical features
df['age_bin'] = pd.cut(df['person_age'], bins=[19, 23, 27, 35, 100], labels=['20-23', '24-27', '28-35', '36+'])
df['income_bin'] = pd.qcut(df['person_income'], q=5, precision=0)
df['emp_length_bin'] = pd.cut(df['person_emp_length'], bins=[-1, 2, 5, 10, 60], labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '10+ yrs'])
df['loan_amnt_bin'] = pd.qcut(df['loan_amnt'], q=4, precision=0)
df['int_rate_bin'] = pd.qcut(df['loan_int_rate'], q=4, precision=0)
df['percent_income_bin'] = pd.cut(df['loan_percent_income'], bins=[-0.01, 0.10, 0.20, 0.30, 1.0], labels=['0-10%', '10-20%', '20-30%', '30%+'])
df['cred_hist_bin'] = pd.cut(df['cb_person_cred_hist_length'], bins=[0, 3, 6, 10, 40], labels=['1-3 yrs', '4-6 yrs', '7-10 yrs', '10+ yrs'])

# Function to display binned default rates cleanly
binned_features = {
    'person_income': 'income_bin',
    'loan_percent_income': 'percent_income_bin',
    'loan_int_rate': 'int_rate_bin',
    'person_emp_length': 'emp_length_bin',
    'loan_amnt': 'loan_amnt_bin',
    'person_age': 'age_bin',
    'cb_person_cred_hist_length': 'cred_hist_bin'
}

for orig_col, bin_col in binned_features.items():
    summary = df.groupby(bin_col, observed=False)['loan_status'].agg(
        Total_Borrowers='count',
        Default_Rate=lambda x: f"{x.mean() * 100:.2f}%"
    ).reset_index()
    print(f"=== Binned Analysis: {orig_col} ===")
    print(summary.to_string(index=False))
    print("\n")

--- Groupby: person_home_ownership ---
  person_home_ownership  total_borrowers  default_count default_rate
0              MORTGAGE            13090           1630       12.45%
1                 OTHER              107             33       30.84%
2                   OWN             2410            167        6.93%
3                  RENT            16072           4995       31.08%


--- Groupby: loan_intent ---
         loan_intent  total_borrowers  default_count default_rate
0  DEBTCONSOLIDATION             5064           1437       28.38%
1          EDUCATION             6288           1066       16.95%
2    HOMEIMPROVEMENT             3510            897       25.56%
3            MEDICAL             5897           1565       26.54%
4           PERSONAL             5367           1046       19.49%
5            VENTURE             5553            814       14.66%


--- Groupby: loan_grade ---
  loan_grade  total_borrowers  default_count default_rate
0          A            10370      